### Use the published autochem workflow to generate DFT features for the compounds in the dataset

NOTE: This notebook needs to be run with the python environment for autoqchem.

In [1]:
from autoqchem.molecule import molecule
from autoqchem.sge_manager import sge_manager
from autoqchem.draw_utils import draw
from autoqchem.db_functions import descriptors
from rdkit import Chem
import pandas as pd
import numpy as np
import logging
logging.basicConfig(level=logging.INFO)


In [ ]:
# connect to UCLA's computation cluster
sm = sge_manager(user='XXXXXX', host='hoffman2.idre.ucla.edu')
sm.connect()

### Load the product smiles strings and create input files for the DFT calculations

In [12]:
# Load the previously saved list of reaction products
df = pd.read_csv("./SB_processed_data.csv",index_col=0)
df

,amine_smiles,acid_smiles,rate
1,Nc1ccc(F)cc1,O=C(Cl)c1ccc(F)cc1,1.292114
3,Nc1ccccc1,O=C(Cl)c1ccc(F)cc1,1.252277
4,Cc1cc(C)c(N)c(C)c1,O=C(Cl)c1ccc(F)cc1,1.204425
5,CCCCCCCc1ccc(N)cc1,O=C(Cl)c1ccc(F)cc1,1.817767
6,CC(C)c1cccc(C(C)C)c1N,O=C(Cl)c1ccc(F)cc1,1.101844
...,...,...,...
1104,N#CCCNc1ccccc1,O=C(Cl)C12CC3CC(CC(C3)C1)C2,0.744872
1105,Nc1ccccc1F,O=C(Cl)C12CC3CC(CC(C3)C1)C2,0.508364
1106,COc1cccc(N)c1,O=C(Cl)C12CC3CC(CC(C3)C1)C2,1.551698
1107,Nc1ccc(C(=O)c2ccccc2)cc1,O=C(Cl)C12CC3CC(CC(C3)C1)C2,1.124939


In [13]:
amine_smiles = list(df["amine_smiles"].unique())
acid_smiles = list(df["acid_smiles"].unique())

# combine the smiles lists
all_smiles = amine_smiles + acid_smiles
print(f"There are {len(amine_smiles)} amines and {len(acid_smiles)}"\
      f" acid chlorides in the dataset ({len(all_smiles)} total compounds).")

There are 28 amines and 31 acid chlorides in the dataset (59 total compounds).


Generate conformers

In [ ]:
# generate molecule objects with up to 8 conformers for each structure
mols = [molecule(s, num_conf=8) for s in all_smiles]

[11:17:01] WARNING: Charges were rearranged



In [ ]:
# create Gaussian jobs locally
for mol in mols:
    sm.create_jobs_for_molecule(mol, theory="APFD",heavy_basis_set="def2tzvp",light_basis_set='def2svp',max_light_atomic_number=10)

INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 1 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 1 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 1 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 7 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 3 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 5 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 7 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 3 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 1 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input file

### Manage the DFT jobs on the cluster

In [ ]:
# Submit jobs
sm.submit_jobs()

INFO:autoqchem.sge_manager:Submitting 0 jobs.


In [ ]:
# Resubmit jobs that did not finish properly
sm.resubmit_incomplete_jobs()

INFO:autoqchem.sge_manager:There are no incomplete jobs to resubmit.


In [ ]:
# Retrieve finished jobs from the cluster
sm.retrieve_jobs()

INFO:autoqchem.sge_manager:There are 0 running/pending jobs, 2 finished jobs.
INFO:autoqchem.sge_manager:Retrieving log files of finished jobs.
100%|██████████| 2/2 [00:01<00:00,  1.39it/s]
INFO:autoqchem.sge_manager:2 jobs finished successfully (all Gaussian steps finished normally). 0 jobs failed.


In [ ]:
# Upload data for finished compounds to the autoqchem database (autoqchem.org)
sm.upload_done_molecules_to_db(tags=["SVR_SchottenBaumann"])

INFO:autoqchem.sge_manager:There are no jobs in done status. Exiting.


### Dataset creation and preprocessing

Download the substrate descriptors

In [14]:
# Download the descriptors for the acid chlorides
data_acid = descriptors(tags=["SVR_SchottenBaumann"],presets=["global","substructure"],conf_option="boltzmann",solvent="None",
                   functional="APFD",basis_set="def2svp",substructure="[C,c]C(=O)Cl")

In [15]:
# Download the descriptors for the amines
# NOTE: the dataset does not contain other NH-containing FGs (e. g. carbamates) so that the provided substructure smarts is sufficient
data_amine = descriptors(tags=["SVR_SchottenBaumann"],presets=["global","substructure"],conf_option="boltzmann",solvent="None",
                   functional="APFD",basis_set="def2svp",substructure="[NH1,NH2]")

In [16]:
# The substrate diisopropylamine was already calculated in another autoqchem dataset (SVR_Amide) with the same settings
#  and was therefore not recalculated for this dataset. Let's also get the descriptors for that compound.

data_dipa = descriptors(tags=["SVR_Amide"],presets=["global","substructure"],conf_option="boltzmann",solvent="None",
                   functional="APFD",basis_set="def2svp",substructure="[NH1]")

In [17]:
# Process the data so that it is in one dataframe
substrate_desc = {}
for substrate in ["amine","acid chloride","amide"]:
    data = data_amine
    if substrate == "acid chloride":
        data = data_acid
    elif substrate == "amide":
        data = data_dipa
    label_dict={}
    for key in data:
        if key != "global":
            # atom descriptor dataframes are by default called atom1, atom2, etc. --> replace with the atom type 
            # and a running number (e. g. "C1" and "C2")
            if data[key].iloc[0,-1] not in label_dict:
                label_dict[data[key].iloc[0,-1]] = 1
            else:
                label_dict[data[key].iloc[0,-1]] += 1
            label = data[key].iloc[0,-1]+str(label_dict[data[key].iloc[0,-1]])
            data[key].drop(columns=["labels","X","Y","Z"],inplace=True)  # drop non-physical features
            data[key].columns = [f"{label}_{column}" for column in data[key].columns]
        else:
            data[key].drop(columns=["converged","multiplicity"],inplace=True)  # same throughout --> not differentiation

    df_combined = pd.concat(data,axis=1)
    df_combined.columns = [multi_column_index[1] for multi_column_index in df_combined.columns]
    substrate_desc[substrate] = df_combined

In [18]:
substrate_desc["amide"]

,E,ES_root_dipole,ES_root_electronic_spatial_extent,ES_root_molar_volume,E_scf,E_thermal_correction,E_zpe,G,G_thermal_correction,H,...,N1_ES_root_NPA_valence,N1_Mulliken_charge,N1_NMR_anisotropy,N1_NMR_shift,N1_NPA_Rydberg,N1_NPA_charge,N1_NPA_core,N1_NPA_total,N1_NPA_valence,N1_VBur
can,,,,,,,,,,,,,,,,,,,,,
C1CCN(C2CCNCC2)CC1,-501.533203,1.198606,2633.371844,1891.238257,-501.838189,0.310304,-501.544485,-501.581479,0.262028,-501.532259,...,5.631209,-0.309093,69.302559,228.411182,0.01487,-0.714497,1.999385,7.714497,5.700243,0.518657
C1CCNC1,-212.121408,2.829700,398.638700,760.644000,-212.253541,0.135034,-212.126398,-212.155085,0.101358,-212.120464,...,5.19921,-0.311719,29.4416,214.1609,0.01435,-0.70502,1.99942,7.70502,5.69124,0.487964
C1COCCN1,-287.214102,2.729200,526.313700,1029.686000,-287.352345,0.140888,-287.219339,-287.247731,0.107259,-287.213158,...,5.32329,-0.311703,68.5394,232.4908,0.01567,-0.71203,1.9994,7.71203,5.69696,0.508968
CC(C)NC(C)C,-291.761604,1.489015,1056.528721,1210.703222,-291.974120,0.215067,-291.770920,-291.804379,0.172291,-291.760660,...,5.207014,-0.290615,44.880436,192.619347,0.016452,-0.735559,1.99931,7.735559,5.719798,0.633051
CC(C)Nc1ccccc1,-404.710988,4.879500,1817.445500,1400.868000,-404.919271,0.212397,-404.720744,-404.755647,0.167738,-404.710044,...,5.44303,-0.20524,66.9462,173.7125,0.01336,-0.65592,1.99915,7.65592,5.6434,0.601164
CC(NC(=O)OCc1ccccc1)C(=O)O,-781.264968,2.200591,4958.801871,1799.809053,-781.508735,0.249729,-781.280029,-781.325666,0.189032,-781.264023,...,5.687487,-0.224572,98.130053,171.801348,0.013666,-0.701527,1.999154,7.701527,5.688706,0.594143
CC1CCCNC1,-290.584436,3.204900,835.622000,1178.903000,-290.776347,0.194411,-290.591371,-290.621744,0.157102,-290.583492,...,5.22925,-0.291462,63.6462,224.9188,0.01507,-0.71103,1.99939,7.71103,5.69657,0.517936
CC1COCCN1,-326.435139,2.021000,749.138850,1052.755000,-326.599473,0.170307,-326.441938,-326.472744,0.132702,-326.434195,...,5.3192,-0.291682,62.722,233.53365,0.01594,-0.7118,1.99938,7.7118,5.69648,0.560494
CCCCN1CCNCC1,-424.263214,2.968330,2280.156454,1418.976607,-424.530291,0.272453,-424.273974,-424.310942,0.224723,-424.262269,...,5.402643,-0.304607,67.915839,230.186373,0.01524,-0.711061,1.999399,7.711061,5.696433,0.512895


In [19]:
# add the descriptors for diisopropylamine to the amine dataframe
dipa_data = substrate_desc["amide"].loc["CC(C)NC(C)C"]
substrate_desc["amine"].loc["CC(C)NC(C)C"] = dipa_data
print(f"Now the amine dataframe has descriptors for {len(substrate_desc['amine'])} substrates.")
print(f"Number of descriptors per amine:", len(substrate_desc["amine"].columns))
print(f"The acid chloride dataframe has descriptors for {len(substrate_desc['acid chloride'])} substrates.")
print(f"Number of descriptors per acid chloride:", len(substrate_desc["acid chloride"].columns))


Now the amine dataframe has descriptors for 28 substrates.
Number of descriptors per amine: 38
The acid chloride dataframe has descriptors for 31 substrates.
Number of descriptors per acid chloride: 86


In [20]:
# combinatorial reaction space
df_reaction = substrate_desc["amine"].merge(substrate_desc["acid chloride"], how="cross")
comb_index = []
for i in substrate_desc["amine"].index:
    for j in substrate_desc["acid chloride"].index:
        comb_index.append(f"{i}.{j}")
df_reaction.index = comb_index

# rename the features
df_reaction.columns = [col.rsplit("_x")[0] + "_amine" if col.endswith("_x") else col for col in df_reaction.columns]
df_reaction.columns = [col.rsplit("_y")[0] + "_COCl" if col.endswith("_y") else col for col in df_reaction.columns] 

df_reaction.head()

,E_amine,ES_root_dipole_amine,ES_root_electronic_spatial_extent_amine,ES_root_molar_volume_amine,E_scf_amine,E_thermal_correction_amine,E_zpe_amine,G_amine,G_thermal_correction_amine,H_amine,...,Cl1_ES_root_NPA_valence,Cl1_Mulliken_charge,Cl1_NMR_anisotropy,Cl1_NMR_shift,Cl1_NPA_Rydberg,Cl1_NPA_charge,Cl1_NPA_core,Cl1_NPA_total,Cl1_NPA_valence,Cl1_VBur
C1CCC(NC2CCCCC2)CC1.CC(=O)Cl,-524.764882,2.280902,3286.519376,1673.698242,-525.112731,0.352139,-524.777177,-524.815637,0.301384,-524.763938,...,6.96776,-0.249833,776.7802,462.7909,0.02158,-0.12005,9.99964,17.12005,7.09883,0.359031
C1CCC(NC2CCCCC2)CC1.CC(C)(C)C(=O)Cl,-524.764882,2.280902,3286.519376,1673.698242,-525.112731,0.352139,-524.777177,-524.815637,0.301384,-524.763938,...,6.96829,-0.243396,657.1407,517.8314,0.02014,-0.11423,9.99964,17.11423,7.09445,0.411737
C1CCC(NC2CCCCC2)CC1.CC(C)(C)c1ccc(C(=O)Cl)cc1,-524.764882,2.280902,3286.519376,1673.698242,-525.112731,0.352139,-524.777177,-524.815637,0.301384,-524.763938,...,6.91221,-0.254743,652.169429,528.632604,0.02076,-0.106427,9.99966,17.106427,7.086012,0.38215
C1CCC(NC2CCCCC2)CC1.CC(C)C(=O)Cl,-524.764882,2.280902,3286.519376,1673.698242,-525.112731,0.352139,-524.777177,-524.815637,0.301384,-524.763938,...,6.970552,-0.246018,714.251382,492.332221,0.020565,-0.118478,9.99964,17.118478,7.098276,0.387452
C1CCC(NC2CCCCC2)CC1.CCC(C(=O)Cl)c1ccccc1,-524.764882,2.280902,3286.519376,1673.698242,-525.112731,0.352139,-524.777177,-524.815637,0.301384,-524.763938,...,7.019298,-0.228389,717.517616,464.585083,0.02103,-0.095897,9.999616,17.095897,7.075249,0.388169


preprocess the descriptors

In [21]:
def feature_preprocessing(df):
    """
    Function for removing non-varied and highly correlated features.
    Take a df as input and returns it in processed form.
    """
    # Remove columns that have only one unique value.
    removed_columns = []
    for column in df.columns:
        if len(np.unique(df[column].values)) < 2:
            removed_columns.append(column)
    df = df.drop(removed_columns, axis=1)
    
    # Remove highly correlated features
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    df = df.drop(to_drop, axis=1)

    # Store the names of the column removed due to correlation
    for column in to_drop:
        removed_columns.append(column)

    print(f"The following features were removed: {removed_columns}")
    
    return df

df_processed = feature_preprocessing(df_reaction)
print(f"There are {len(df_processed.columns)} final descriptors.")
df_processed.head()

The following features were removed: ['charge_amine', 'charge_COCl', 'E_scf_amine', 'E_zpe_amine', 'G_amine', 'G_thermal_correction_amine', 'H_amine', 'H_thermal_correction_amine', 'electronic_spatial_extent_amine', 'lumo_energy_amine', 'number_of_atoms_amine', 'zero_point_correction_amine', 'N1_ES_root_NPA_total', 'N1_ES_root_NPA_valence', 'N1_NPA_total', 'N1_NPA_valence', 'E_scf_COCl', 'E_zpe_COCl', 'G_COCl', 'G_thermal_correction_COCl', 'H_COCl', 'H_thermal_correction_COCl', 'electronic_spatial_extent_COCl', 'number_of_atoms_COCl', 'zero_point_correction_COCl', 'C1_ES_root_NPA_total', 'C1_ES_root_NPA_valence', 'C1_Mulliken_charge', 'C1_NPA_Rydberg', 'C1_NPA_charge', 'C1_NPA_core', 'C1_NPA_total', 'C1_NPA_valence', 'C2_ES_root_NPA_total', 'C2_ES_root_NPA_valence', 'C2_NPA_total', 'C2_NPA_valence', 'O1_APT_charge', 'O1_ES_root_NPA_Rydberg', 'O1_ES_root_NPA_charge', 'O1_ES_root_NPA_total', 'O1_ES_root_NPA_valence', 'O1_NPA_core', 'O1_NPA_total', 'O1_NPA_valence', 'Cl1_ES_root_Mulliken_

,E_amine,ES_root_dipole_amine,ES_root_electronic_spatial_extent_amine,ES_root_molar_volume_amine,E_thermal_correction_amine,dipole_amine,electronegativity_amine,hardness_amine,homo_energy_amine,molar_mass_amine,...,O1_NPA_charge,O1_VBur,Cl1_APT_charge,Cl1_Mulliken_charge,Cl1_NMR_anisotropy,Cl1_NMR_shift,Cl1_NPA_Rydberg,Cl1_NPA_charge,Cl1_NPA_core,Cl1_VBur
C1CCC(NC2CCCCC2)CC1.CC(=O)Cl,-524.764882,2.280902,3286.519376,1673.698242,0.352139,0.8732,0.0799,0.13752,-0.21742,181.3204,...,-0.48574,0.347231,-0.497062,-0.249833,776.7802,462.7909,0.02158,-0.12005,9.99964,0.359031
C1CCC(NC2CCCCC2)CC1.CC(C)(C)C(=O)Cl,-524.764882,2.280902,3286.519376,1673.698242,0.352139,0.8732,0.0799,0.13752,-0.21742,181.3204,...,-0.49442,0.412996,-0.486258,-0.243396,657.1407,517.8314,0.02014,-0.11423,9.99964,0.411737
C1CCC(NC2CCCCC2)CC1.CC(C)(C)c1ccc(C(=O)Cl)cc1,-524.764882,2.280902,3286.519376,1673.698242,0.352139,0.8732,0.0799,0.13752,-0.21742,181.3204,...,-0.50349,0.38578,-0.51556,-0.254743,652.169429,528.632604,0.02076,-0.106427,9.99966,0.38215
C1CCC(NC2CCCCC2)CC1.CC(C)C(=O)Cl,-524.764882,2.280902,3286.519376,1673.698242,0.352139,0.8732,0.0799,0.13752,-0.21742,181.3204,...,-0.492762,0.39244,-0.495845,-0.246018,714.251382,492.332221,0.020565,-0.118478,9.99964,0.387452
C1CCC(NC2CCCCC2)CC1.CCC(C(=O)Cl)c1ccccc1,-524.764882,2.280902,3286.519376,1673.698242,0.352139,0.8732,0.0799,0.13752,-0.21742,181.3204,...,-0.500502,0.434351,-0.476558,-0.228389,717.517616,464.585083,0.02103,-0.095897,9.999616,0.388169


Assign the rate data

In [22]:
df.head()

,amine_smiles,acid_smiles,rate
1,Nc1ccc(F)cc1,O=C(Cl)c1ccc(F)cc1,1.292114
3,Nc1ccccc1,O=C(Cl)c1ccc(F)cc1,1.252277
4,Cc1cc(C)c(N)c(C)c1,O=C(Cl)c1ccc(F)cc1,1.204425
5,CCCCCCCc1ccc(N)cc1,O=C(Cl)c1ccc(F)cc1,1.817767
6,CC(C)c1cccc(C(C)C)c1N,O=C(Cl)c1ccc(F)cc1,1.101844


In [23]:
df_processed["rate"] = np.nan
for idx in df_processed.index:
    [amine, cocl] = idx.split(".")
    df_processed.loc[idx,"rate"] = df.loc[(df["amine_smiles"] == amine) & (df["acid_smiles"] == cocl), "rate"].values[0]

# Check that all rates were assigned
print("Number of reactions without an assigned rate:",df_processed["rate"].isna().sum())


Number of reactions without an assigned rate: 0


In [24]:
# Save the dataset
df_processed.to_csv("./SB_dset.csv",index=True,header=True)

Make separate datasets for just the amine and acid chlorides descriptors

In [ ]:
amine_data = substrate_desc["amine"].copy()
acid_data = substrate_desc["acid chloride"].copy()

In [28]:
print("Working on amine descriptors")
amine_data = feature_preprocessing(amine_data)
print(f"There are {len(amine_data.columns)} final descriptors after preprocessing. Saving descriptors.")
amine_data.to_csv("SB_dset_just_amines.csv",index=True,header=True)

print("Working on acid chloride descriptors")
acid_data = feature_preprocessing(acid_data)
print(f"There are {len(acid_data.columns)} final descriptors after preprocessing. Saving descriptors.")
acid_data.to_csv("SB_dset_just_acid_chlorides.csv",index=True,header=True)

Working on amine descriptors
The following features were removed: ['charge', 'E_scf', 'E_zpe', 'G', 'G_thermal_correction', 'H', 'H_thermal_correction', 'electronic_spatial_extent', 'lumo_energy', 'number_of_atoms', 'zero_point_correction', 'N1_ES_root_NPA_total', 'N1_ES_root_NPA_valence', 'N1_NPA_total', 'N1_NPA_valence']
There are 23 final descriptors after preprocessing. Saving descriptors.
Working on acid chloride descriptors
The following features were removed: ['charge', 'E_scf', 'E_zpe', 'G', 'G_thermal_correction', 'H', 'H_thermal_correction', 'electronic_spatial_extent', 'number_of_atoms', 'zero_point_correction', 'C1_ES_root_NPA_total', 'C1_ES_root_NPA_valence', 'C1_Mulliken_charge', 'C1_NPA_Rydberg', 'C1_NPA_charge', 'C1_NPA_core', 'C1_NPA_total', 'C1_NPA_valence', 'C2_ES_root_NPA_total', 'C2_ES_root_NPA_valence', 'C2_NPA_total', 'C2_NPA_valence', 'O1_APT_charge', 'O1_ES_root_NPA_Rydberg', 'O1_ES_root_NPA_charge', 'O1_ES_root_NPA_total', 'O1_ES_root_NPA_valence', 'O1_NPA_cor